In [1]:
import sys
import pandas as pd
from pathlib import Path

In [2]:
current_dir = Path.cwd()
utils_path = next(
    (
        p
        for p in [current_dir] + list(current_dir.parents)
        if (p / "notebook_utils.py").exists()
    ),
    None,
)

if utils_path is None:
    raise FileNotFoundError("notebook_utils.py не найден!")

sys.path.append(str(utils_path))

In [3]:
from notebook_utils import (
    setup_env,
    load_data_for_modeling,
    get_exp_manager
)

PROJECT_ROOT, config = setup_env()
exp_manager = get_exp_manager()

print(f"Project Root: {PROJECT_ROOT}")

Project Root: D:\Education\Arcticle\02_dtp_project_JAER


## ДТП датасет

In [4]:
df_dtp = load_data_for_modeling(
    config, 
    PROJECT_ROOT, 
    data_source="full_df"
)

print(f"DTP Dataset shape: {df_dtp.shape}")

Загрузка данных из: D:\Education\Arcticle\02_dtp_project_JAER\data\processed\dtp_full_dataset.parquet
Режим таргета: binary_severe. Распределение:
target
0    0.566
1    0.434
Name: proportion, dtype: float64
DTP Dataset shape: (1465882, 89)


In [5]:
print(df_dtp[['region_id']].nunique())

region_id    2237
dtype: int64


In [6]:
sorted(df_dtp['year'].unique())

[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

In [20]:
df_dtp['region_id'].nunique()

2237

## Экономический датасет

In [7]:
econ_path = PROJECT_ROOT / config['paths']['econom_data']
df_econ = pd.read_excel(econ_path)
print(f"shape: {df_econ.shape}")

shape: (1000, 13)


In [8]:
df_econ.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   region_name    1000 non-null   object 
 1   mun_district   1000 non-null   object 
 2   municipality   1000 non-null   object 
 3   year           1000 non-null   int64  
 4   Population     1000 non-null   float64
 5   GAP            1000 non-null   float64
 6   Avg_Salary     1000 non-null   float64
 7   Employment     1000 non-null   float64
 8   Street_Length  1000 non-null   float64
 9   Land_Area      1000 non-null   float64
 10  RTA_dead       1000 non-null   object 
 11  RTA_serious    1000 non-null   object 
 12  RTA_minor      1000 non-null   object 
dtypes: float64(6), int64(1), object(6)
memory usage: 101.7+ KB


In [9]:
print(df_econ.sample(1))

    region_name                                       mun_district  \
397      Москва  Муниципальные образования города Москвы (столи...   

     municipality  year  Population      GAP  Avg_Salary  Employment  \
397  Город Москва  2021     12645.3  12687.0    112768.3   5160600.0   

     Street_Length  Land_Area RTA_dead RTA_serious RTA_minor  
397         6253.4   256150.0      328        1586      6542  


In [10]:
df_econ.head()

,region_name,mun_district,municipality,year,Population,GAP,Avg_Salary,Employment,Street_Length,Land_Area,RTA_dead,RTA_serious,RTA_minor
0,Иркутская область,Ангарский,Ангарск,2014,238.9,102.2,36376.9,61818.0,414.9,114872.2,Н/Д,Н/Д,Н/Д
1,Иркутская область,Ангарский,Ангарск,2015,239.2,102.5,38990.9,59275.0,414.9,114872.2,21,74,65
2,Иркутская область,Ангарский,Ангарск,2016,238.7,97.4,41307.6,55846.0,454.3,114872.2,10,99,84
3,Иркутская область,Ангарский,Ангарск,2017,238.3,92.2,43217.9,55112.0,459.7,114872.2,14,91,94
4,Иркутская область,Ангарский,Ангарск,2018,237.9,93.8,47401.3,53541.0,415.0,114872.2,5,68,77


In [11]:
sorted(df_econ['year'].unique())

[2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

In [ ]:
df_econ.columns.tolist()


['region_name',
 'mun_district',
 'municipality',
 'year',
 'Population',
 'GAP',
 'Avg_Salary',
 'Employment',
 'Street_Length',
 'Land_Area',
 'RTA_dead',
 'RTA_serious',
 'RTA_minor']

In [16]:
df_econ['region_name'].unique()

array(['Иркутская область', 'Московская область', 'Амурская область',
       'Новгородская область', 'Приморский край', 'Волгоградская область',
       'Республика Хакасия', 'Краснодарский край',
       'Архангельская область без автономного округа',
       'Астраханская область', 'Саратовская область', 'Алтайский край',
       'Белгородская область', 'Брянская область',
       'Республика Северная Осетия — Алания', 'Владимирская область',
       'Ростовская область', 'Вологодская область', 'Воронежская область',
       'Чеченская Республика', 'Нижегородская область',
       'Свердловская область', 'Удмуртская Республика',
       'Республика Марий Эл', 'Республика Татарстан',
       'Калининградская область', 'Калужская область',
       'Кировская область', 'Хабаровский край', 'Красноярский край',
       'Курганская область', 'Курская область', 'Липецкая область',
       'Республика Дагестан', 'Москва', 'Мурманская область',
       'Новосибирская область', 'Омская область', 'Орловская 

### Пропуски

In [13]:
econ_numeric_cols = [
    'Population', 'GAP', 'Avg_Salary', 'Employment', 
    'Street_Length', 'Land_Area', 'RTA_dead', 'RTA_serious', 'RTA_minor'
]

for col in econ_numeric_cols:
    if col in df_econ.columns:
        if df_econ[col].dtype == 'object':
            df_econ[col] = df_econ[col].astype(str).str.replace(',', '.')
        
        df_econ[col] = pd.to_numeric(df_econ[col], errors='coerce')
        
        median_val = df_econ[col].median()
        
        if df_econ[col].isna().sum() > 0:
            print(f"   Колонка '{col}': заменено {df_econ[col].isna().sum()} значений 'Н/Д' на медиану ({median_val:.1f})")
            df_econ[col] = df_econ[col].fillna(median_val)

print("Типы данных после очистки:")
print(df_econ[econ_numeric_cols].dtypes)

   Колонка 'RTA_dead': заменено 104 значений 'Н/Д' на медиану (19.0)
   Колонка 'RTA_serious': заменено 103 значений 'Н/Д' на медиану (105.0)
   Колонка 'RTA_minor': заменено 103 значений 'Н/Д' на медиану (298.0)
Типы данных после очистки:
Population       float64
GAP              float64
Avg_Salary       float64
Employment       float64
Street_Length    float64
Land_Area        float64
RTA_dead         float64
RTA_serious      float64
RTA_minor        float64
dtype: object


### Очистка регионов

In [14]:
import pandas as pd
import re

def clean_econ_region(text):
    """
    Очищает название региона от административных терминов.
    """
    if pd.isna(text):
        return ""

    text = str(text).lower()
    
    garbage_words = [
        'автономный округ', 'автономная область', 
        'республика', 'область', 'край', 'алания', 'петербург', 'мансийский',
        'г.', 'ао', 'эл', 'югра', 'без автономного округа',
        ' — ', ' - ', '-', '—'
    ]
    
    for word in garbage_words:
        text = text.replace(word, ' ')
        
    text = re.sub(r'\(.*?\)', '', text)
    text = text.replace('.', '').replace(',', '').replace('-', '_').replace(' ', '')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df_econ['clean_name'] = df_econ['region_name'].apply(clean_econ_region)

In [15]:
print(len(sorted(df_econ['region_name'].unique())))
print(len(sorted(df_econ['clean_name'].unique())))

68
68
